# Algebraic Reconstruction of Isospectral Unfoldings

Given a scalar rational function $r(\lambda) = a_{ss} - \mathbf{A}_{S\bar{S}}(A_{\bar{S}\bar{S}} - \lambda I)^{-1}\mathbf{A}_{\bar{S}S}$, we want to find **all** graphs $G$ whose isospectral reduction to the single vertex $s \in S$ equals $r(\lambda)$. Such graphs are called **admissible unfoldings** of $r$.

## The algebraic structure

For $|S| = 1$, $G$ consists of the kept vertex $s$ together with $k = |\bar{S}|$ complement vertices. Let $A_H = A_{\bar{S}\bar{S}}$ be the adjacency of the complement subgraph $H$, and let $\mathbf{c} = A_{\bar{S}S}$ be the binary coupling vector between $\bar{S}$ and $s$. Then:

$$r(\lambda) = a_{ss} - \mathbf{c}^T (A_H - \lambda I)^{-1} \mathbf{c}$$

Since $a_{ss} = 0$ for simple graphs (no self-loops):

$$r(\lambda) = -\mathbf{c}^T (A_H - \lambda I)^{-1} \mathbf{c}$$

## Algorithm

1. **Enumerate complement graphs**: For each graph $H$ in the NetworkX graph atlas (all simple graphs on $\leq 7$ vertices), compute the symbolic matrix $(A_H - \lambda I)^{-1}$.
2. **Enumerate coupling vectors**: For each binary vector $\mathbf{c} \in \{0,1\}^k \setminus \{\mathbf{0}\}$ (the zero vector corresponds to disconnected graphs — excluded):
   - Compute $f(\lambda) = -\mathbf{c}^T (A_H - \lambda I)^{-1} \mathbf{c}$ symbolically.
   - Check if $f(\lambda) = r(\lambda)$ by simplifying the difference.
3. **Assemble and verify**: For each matching $(H, \mathbf{c})$, build the full adjacency matrix and verify admissibility.
4. **Deduplicate**: Keep only non-isomorphic full graphs.

**Limitation**: The graph atlas only covers graphs up to 7 vertices. For a reduction from an $n$-vertex graph keeping 1 vertex, the complement has $n-1$ vertices — so this approach works exactly for original graphs up to $n = 8$, and finds *partial* results (smaller unfoldings) for larger graphs.


In [1]:
import numpy as np
import networkx as nx
import itertools
import time
from collections import defaultdict

try:
    import sympy as sp
    print("SymPy available:", sp.__version__)
except ImportError:
    sp = None
    print("SymPy not available — install sympy")

lam = sp.symbols('lambda') if sp else None
print("Setup complete.")

SymPy available: 1.14.0
Setup complete.


## Step 1 — Compute the isospectral reduction $r(\lambda)$

Given a graph $G$ and a kept vertex set $S = \{s\}$, compute the scalar rational function:

$$r(\lambda) = -\mathbf{c}^T (A_H - \lambda I)^{-1} \mathbf{c}$$

where $A_H$ is the adjacency of the complement $\bar{S}$ and $\mathbf{c} = A_{\bar{S}S}$ is the coupling column.

We also need this function as the **target** when searching for all admissible unfoldings.


In [2]:
def compute_reduction(G, kept_vertex, lam):
    """
    Compute the isospectral reduction r(lambda) of graph G to the single vertex s.

    Returns sp.Expr (a simplified rational function in lambda).
    """
    n = G.number_of_nodes()
    nodes = list(G.nodes())
    s = kept_vertex
    Sbar = [v for v in nodes if v != s]

    A = sp.Matrix(nx.to_numpy_array(G, nodelist=nodes, dtype=int).tolist())
    idx = {v: i for i, v in enumerate(nodes)}
    s_idx = idx[s]
    Sbar_idx = [idx[v] for v in Sbar]

    a_ss = A[s_idx, s_idx]
    c = sp.Matrix([A[i, s_idx] for i in Sbar_idx])
    A_H = sp.Matrix([[A[i, j] for j in Sbar_idx] for i in Sbar_idx])

    M = A_H - lam * sp.eye(len(Sbar))
    r = sp.cancel(sp.together(a_ss - (c.T * M.inv() * c)[0, 0]))
    return r


# Sanity check: P_3 reduced to middle vertex should give r = 2/lambda
if sp:
    G_p3 = nx.path_graph(3)
    r_p3 = compute_reduction(G_p3, 1, lam)
    print("P_3 reduction to middle vertex (vertex 1):", r_p3)
    assert sp.simplify(r_p3 - 2/lam) == 0, "Sanity check failed"
    print("  Sanity check passed: r = 2/lambda")

P_3 reduction to middle vertex (vertex 1): 2/lambda
  Sanity check passed: r = 2/lambda


## Step 2 — Enumerate all (H, c) pairs matching the target reduction

For each graph $H$ in the NetworkX atlas (up to `n_atlas_max` vertices) and each nonzero binary coupling
vector $\mathbf{c} \in \{0,1\}^k \setminus \{\mathbf{0}\}$:

1. Compute $f(\lambda) = -\mathbf{c}^T (A_H - \lambda I)^{-1} \mathbf{c}$ symbolically.
2. Check $f(\lambda) = r(\lambda)$ by computing $\mathtt{cancel}(f - r)$ and testing if it is zero.

**Key insight:** We precompute $(A_H - \lambda I)^{-1}$ once per graph $H$, then evaluate the quadratic form
for each of the $2^k - 1$ coupling vectors. Across all atlas graphs, this is feasible for $k \leq 7$.

**Note on cancellation:** The rational function $r(\lambda)$ may simplify (e.g., $2/\lambda$ from P\_3 middle vertex,
even though the complement has 2 vertices). Reading $k$ from $\deg(\text{denominator})$ would be wrong.
Instead we directly test all $(H, \mathbf{c})$ pairs — no denominator parsing needed.


In [3]:
def enumerate_unfoldings(target_r, lam, n_atlas_max=7, verbose=True):
    """
    Find all (H, c) pairs from the graph atlas such that
        -c^T (A_H - lam*I)^{-1} c == target_r

    Parameters
    ----------
    target_r   : sp.Expr   — target rational function (already simplified)
    lam        : sp.Symbol — the lambda symbol
    n_atlas_max: int       — max complement size to search (<=7 for graph atlas)
    verbose    : bool

    Returns list of dicts with keys: 'H' (nx.Graph), 'c' (tuple of ints), 'k' (int)
    """
    target = sp.cancel(sp.together(target_r))
    results = []
    n_graphs_checked = 0

    for G0 in nx.graph_atlas_g():
        k = G0.number_of_nodes()
        if k == 0:
            continue
        if k > n_atlas_max:
            break

        G = nx.convert_node_labels_to_integers(G0)
        if nx.number_of_selfloops(G) > 0:
            continue

        A = sp.Matrix(nx.to_numpy_array(G, dtype=int).tolist())
        M = A - lam * sp.eye(k)
        try:
            M_inv = M.inv()
        except Exception:
            continue  # singular symbolically (shouldn't happen for variable lam)

        n_graphs_checked += 1

        for c_tuple in itertools.product([0, 1], repeat=k):
            if all(x == 0 for x in c_tuple):
                continue  # all-zero c → no coupling → s is isolated

            c = sp.Matrix(list(c_tuple))
            f = sp.cancel(sp.together(-(c.T * M_inv * c)[0, 0]))

            if sp.simplify(f - target) == 0:
                results.append({'H': G, 'c': c_tuple, 'k': k})
                if verbose:
                    print(f"  Match: k={k}, H edges={list(G.edges())}, c={c_tuple}")

    if verbose:
        print(f"Checked {n_graphs_checked} atlas graphs (n<=atlas_max={n_atlas_max}). "
              f"Found {len(results)} (H,c) pairs.")

    return results


# Quick test: P_3 middle vertex, expect at least (H=2 isolated, c=(1,1))
if sp:
    print("Testing enumerate_unfoldings on P_3 (middle vertex)...")
    pairs_p3 = enumerate_unfoldings(r_p3, lam, n_atlas_max=3, verbose=True)
    print(f"Found {len(pairs_p3)} pairs (k<=3)")

Testing enumerate_unfoldings on P_3 (middle vertex)...
  Match: k=2, H edges=[], c=(1, 1)
  Match: k=3, H edges=[], c=(0, 1, 1)
  Match: k=3, H edges=[], c=(1, 0, 1)
  Match: k=3, H edges=[], c=(1, 1, 0)
Checked 7 atlas graphs (n<=atlas_max=3). Found 4 (H,c) pairs.
Found 4 pairs (k<=3)


## Step 3 — Assemble the full graph and verify admissibility

For each matching $(H, \mathbf{c})$:

- Build the adjacency matrix $A$ of the full graph $G = s \cup V(H)$:
  - Vertex 0 is the kept vertex $s$.
  - Vertices $1, \ldots, k$ are the complement vertices (from $H$).
  - $A_{0, i+1} = A_{i+1, 0} = c_i$ (coupling).
  - $A_{i+1, j+1} = (A_H)_{i,j}$ (internal edges of complement).
- **Verify admissibility**: symbolically compute the isospectral reduction of this full graph back to vertex 0 and confirm it equals $r(\lambda)$.
- **Check graph validity**: simple (no self-loops), undirected, binary.

Finally, **deduplicate** by graph isomorphism: two unfoldings are the same if their full adjacency graphs are isomorphic (possibly with different vertex labelings).


In [4]:
def assemble_unfolding(H, c_tuple):
    """
    Build the full (k+1)x(k+1) adjacency matrix.
    Vertex 0 = kept vertex s; vertices 1..k = complement V(H).
    """
    k = H.number_of_nodes()
    A = np.zeros((k + 1, k + 1), dtype=int)
    # Coupling: s ↔ complement
    for i, ci in enumerate(c_tuple):
        A[0, i + 1] = ci
        A[i + 1, 0] = ci
    # Internal complement edges
    A_H = nx.to_numpy_array(H, dtype=int)
    A[1:, 1:] = A_H
    return A


def isospectral_reduction_check(A_full, kept_idx, target_r, lam):
    """
    Symbolically compute the isospectral reduction of A_full to kept_idx,
    and return True if it equals target_r.
    """
    n = A_full.shape[0]
    A_sym = sp.Matrix(A_full.tolist())
    S = [kept_idx]
    Sbar = [i for i in range(n) if i not in S]

    a_ss = A_sym[kept_idx, kept_idx]
    c = sp.Matrix([A_sym[i, kept_idx] for i in Sbar])
    A_H = sp.Matrix([[A_sym[i, j] for j in Sbar] for i in Sbar])
    M = A_H - lam * sp.eye(len(Sbar))
    r = sp.cancel(sp.together(a_ss - (c.T * M.inv() * c)[0, 0]))
    return sp.simplify(r - target_r) == 0


def is_valid_graph(A):
    """Check simple, undirected, binary adjacency matrix."""
    n = A.shape[0]
    if not np.array_equal(A, A.T):
        return False
    if np.any(np.diag(A) != 0):
        return False
    if not np.all((A == 0) | (A == 1)):
        return False
    return True


def deduplicate_graphs(A_list):
    """
    Given a list of numpy adjacency matrices, return one representative per
    isomorphism class using networkx graph isomorphism.
    """
    reps = []
    for A in A_list:
        G = nx.from_numpy_array(A)
        is_new = True
        for rep in reps:
            if nx.is_isomorphic(G, nx.from_numpy_array(rep)):
                is_new = False
                break
        if is_new:
            reps.append(A)
    return reps


# Sanity check: assemble P_3 from (H=K_2^bar, c=(1,1)) and verify
if sp:
    H2 = nx.empty_graph(2)
    A_assembled = assemble_unfolding(H2, (1, 1))
    print("Assembled adjacency for (H=2 isolated, c=(1,1)):")
    print(A_assembled)
    ok = isospectral_reduction_check(A_assembled, kept_idx=0, target_r=r_p3, lam=lam)
    print(f"Admissibility check: {ok}  (expected True)")

Assembled adjacency for (H=2 isolated, c=(1,1)):
[[0 1 1]
 [1 0 0]
 [1 0 0]]
Admissibility check: True  (expected True)


## Main Pipeline

Putting it all together: given any graph $G$ and kept vertex $s$,

1. Compute $r(\lambda)$ using `compute_reduction`.
2. Find all $(H, \mathbf{c})$ pairs via `enumerate_unfoldings`.
3. Assemble and verify each candidate via `assemble_unfolding` + `isospectral_reduction_check`.
4. Deduplicate by isomorphism.

The result is the **complete set of non-isomorphic admissible unfoldings** up to the atlas size limit.


In [5]:
def reconstruct_all_unfoldings(
    G_source,
    kept_vertex,
    lam,
    n_atlas_max=7,
    verbose=True,
):
    """
    Find all non-isomorphic admissible unfoldings of the reduction
    r(lambda) = isospec_reduction(G_source, kept_vertex).

    Returns
    -------
    list of dicts: {'A': np.ndarray, 'H': nx.Graph, 'c': tuple, 'n': int,
                    'is_original': bool, 'verified': bool}
    """
    t0 = time.time()

    # Step 1: compute target reduction
    target_r = compute_reduction(G_source, kept_vertex, lam)
    if verbose:
        print(f"Target r(lambda) = {target_r}")

    # Step 2: enumerate (H, c) pairs
    if verbose:
        print(f"Searching atlas (n<={n_atlas_max})...")
    pairs = enumerate_unfoldings(target_r, lam, n_atlas_max=n_atlas_max, verbose=verbose)

    if not pairs:
        if verbose:
            print("No (H,c) pairs found.")
        return []

    # Step 3: assemble and verify
    valid_As = []
    full_results = []
    for rec in pairs:
        H, c_tuple, k = rec['H'], rec['c'], rec['k']
        A = assemble_unfolding(H, c_tuple)
        if not is_valid_graph(A):
            continue
        # Optionally verify symbolically (slow — skip by default for large k)
        verified = True  # already confirmed by enumerate_unfoldings
        full_results.append({'A': A, 'H': H, 'c': c_tuple, 'n': k + 1,
                              'k': k, 'verified': verified})
        valid_As.append(A)

    # Step 4: deduplicate
    unique_As = deduplicate_graphs(valid_As)
    unique_results = []
    for uA in unique_As:
        for rec in full_results:
            if np.array_equal(rec['A'], uA):
                unique_results.append(rec)
                break

    dt = time.time() - t0
    if verbose:
        print(f"\nDone in {dt:.1f}s: {len(pairs)} (H,c) pairs → "
              f"{len(unique_results)} non-isomorphic unfoldings.")

    return unique_results

## Demo 1 — P₃ (path on 3 vertices, keep middle vertex)

This is a known case: $P_3$ reduced to its middle vertex gives $r(\lambda) = 2/\lambda$.
We expect to recover $P_3$ itself as the canonical unfolding, plus potentially other
equivalent graphs (other unfoldings with the same reduction).

Note: The reduction $2/\lambda$ has denominator of degree 1, but the complement of the
middle vertex in $P_3$ has 2 vertices. This *cancellation* is why we cannot read $k$
from the denominator degree — we must test all $(H, \mathbf{c})$ pairs directly.


In [6]:
if sp:
    print("=" * 60)
    print("Demo 1: Reconstruct unfoldings for P_3 (middle vertex kept)")
    print(f"Target reduction: r(lam) = {r_p3}")
    print("=" * 60)

    res_p3 = reconstruct_all_unfoldings(
        G_source=nx.path_graph(3),
        kept_vertex=1,
        lam=lam,
        n_atlas_max=4,   # search complements up to 4 vertices
        verbose=True,
    )

    print(f"\nNon-isomorphic unfoldings (up to atlas n=4):")
    for rec in res_p3:
        G_full = nx.from_numpy_array(rec['A'])
        print(f"  n={rec['n']}, k={rec['k']}, "
              f"H_edges={list(rec['H'].edges())}, c={rec['c']}, "
              f"G_connected={nx.is_connected(G_full)}")

Demo 1: Reconstruct unfoldings for P_3 (middle vertex kept)
Target reduction: r(lam) = 2/lambda
Target r(lambda) = 2/lambda
Searching atlas (n<=4)...
  Match: k=2, H edges=[], c=(1, 1)
  Match: k=3, H edges=[], c=(0, 1, 1)
  Match: k=3, H edges=[], c=(1, 0, 1)
  Match: k=3, H edges=[], c=(1, 1, 0)
  Match: k=4, H edges=[], c=(0, 0, 1, 1)
  Match: k=4, H edges=[], c=(0, 1, 0, 1)
  Match: k=4, H edges=[], c=(0, 1, 1, 0)
  Match: k=4, H edges=[], c=(1, 0, 0, 1)
  Match: k=4, H edges=[], c=(1, 0, 1, 0)
  Match: k=4, H edges=[], c=(1, 1, 0, 0)
  Match: k=4, H edges=[(2, 3)], c=(1, 1, 0, 0)
Checked 18 atlas graphs (n<=atlas_max=4). Found 11 (H,c) pairs.

Done in 3.6s: 11 (H,c) pairs → 4 non-isomorphic unfoldings.

Non-isomorphic unfoldings (up to atlas n=4):
  n=3, k=2, H_edges=[], c=(1, 1), G_connected=True
  n=4, k=3, H_edges=[], c=(0, 1, 1), G_connected=False
  n=5, k=4, H_edges=[], c=(0, 0, 1, 1), G_connected=False
  n=5, k=4, H_edges=[(2, 3)], c=(1, 1, 0, 0), G_connected=False


## Demo 2 — P₅ (path on 5 vertices, keep one end vertex)

$P_5$ reduced to an end vertex gives a non-trivial rational function. The complement
subgraph has 4 vertices. We search complements up to 5 vertices to find both the
original structure and any equivalent unfoldings.

**Expected**: At least one unfolding isomorphic to $P_5$ itself is recovered.


In [7]:
if sp:
    print("=" * 60)
    print("Demo 2: Reconstruct unfoldings for P_5 (end vertex kept)")
    print("=" * 60)

    G_p5 = nx.path_graph(5)
    r_p5 = compute_reduction(G_p5, kept_vertex=0, lam=lam)
    print(f"Target r(lambda) = {r_p5}")

    res_p5 = reconstruct_all_unfoldings(
        G_source=G_p5,
        kept_vertex=0,
        lam=lam,
        n_atlas_max=5,   # complement has 4 vertices, search up to 5
        verbose=True,
    )

    print(f"\nNon-isomorphic unfoldings found: {len(res_p5)}")
    original_recovered = False
    for rec in res_p5:
        G_full = nx.from_numpy_array(rec['A'])
        is_orig = nx.is_isomorphic(G_full, G_p5)
        if is_orig:
            original_recovered = True
        print(f"  n={rec['n']}, H_edges={list(rec['H'].edges())}, c={rec['c']}, "
              f"iso_to_P5={is_orig}")
    print(f"\nOriginal P_5 recovered: {original_recovered}")

Demo 2: Reconstruct unfoldings for P_5 (end vertex kept)
Target r(lambda) = (lambda**3 - 2*lambda)/(lambda**4 - 3*lambda**2 + 1)
Target r(lambda) = (lambda**3 - 2*lambda)/(lambda**4 - 3*lambda**2 + 1)
Searching atlas (n<=5)...
  Match: k=4, H edges=[(0, 1), (0, 3), (1, 2)], c=(0, 0, 0, 1)
  Match: k=4, H edges=[(0, 1), (0, 3), (1, 2)], c=(0, 0, 1, 0)
  Match: k=5, H edges=[(0, 4), (2, 3), (3, 4)], c=(0, 0, 1, 0, 0)
  Match: k=5, H edges=[(0, 4), (2, 3), (3, 4)], c=(1, 0, 0, 0, 0)
Checked 52 atlas graphs (n<=atlas_max=5). Found 4 (H,c) pairs.

Done in 64.1s: 4 (H,c) pairs → 2 non-isomorphic unfoldings.

Non-isomorphic unfoldings found: 2
  n=5, H_edges=[(0, 1), (0, 3), (1, 2)], c=(0, 0, 0, 1), iso_to_P5=True
  n=6, H_edges=[(0, 4), (2, 3), (3, 4)], c=(0, 0, 1, 0, 0), iso_to_P5=False

Original P_5 recovered: True


## Demo 3 — Random ER graph (n=5, kept vertex)

We now test on a random Erdős-Rényi graph. Since the graph atlas only covers up to 7 vertices,
we use $n=5$ so the complement has 4 vertices and is fully covered by the atlas.

The key question: does the algorithm recover the original graph, and are there other admissible
unfoldings?


In [8]:
if sp:
    print("=" * 60)
    print("Demo 3: Random ER graph n=5, p=0.5, keep vertex 0")
    print("=" * 60)

    rng = np.random.RandomState(42)
    G_er = nx.erdos_renyi_graph(5, 0.5, seed=7)
    while not nx.is_connected(G_er):
        G_er = nx.erdos_renyi_graph(5, 0.5, seed=rng.randint(10000))

    print(f"Graph edges: {sorted(G_er.edges())}")
    kept = 0
    r_er = compute_reduction(G_er, kept_vertex=kept, lam=lam)
    print(f"Target r(lambda) = {r_er}")

    res_er = reconstruct_all_unfoldings(
        G_source=G_er,
        kept_vertex=kept,
        lam=lam,
        n_atlas_max=5,
        verbose=True,
    )

    print(f"\nNon-isomorphic unfoldings found: {len(res_er)}")
    original_recovered = False
    for rec in res_er:
        G_full = nx.from_numpy_array(rec['A'])
        is_orig = nx.is_isomorphic(G_full, G_er)
        if is_orig:
            original_recovered = True
        print(f"  n={rec['n']}, H_edges={list(rec['H'].edges())}, c={rec['c']}, "
              f"iso_to_original={is_orig}")
    print(f"\nOriginal ER graph recovered: {original_recovered}")

Demo 3: Random ER graph n=5, p=0.5, keep vertex 0
Graph edges: [(0, 1), (0, 2), (0, 4), (1, 3), (1, 4), (2, 4), (3, 4)]
Target r(lambda) = (3*lambda**3 + 4*lambda**2 - 2*lambda - 2)/(lambda**4 - 4*lambda**2 - 2*lambda + 1)
Target r(lambda) = (3*lambda**3 + 4*lambda**2 - 2*lambda - 2)/(lambda**4 - 4*lambda**2 - 2*lambda + 1)
Searching atlas (n<=5)...
  Match: k=4, H edges=[(0, 3), (1, 2), (1, 3), (2, 3)], c=(1, 0, 1, 1)
  Match: k=4, H edges=[(0, 3), (1, 2), (1, 3), (2, 3)], c=(1, 1, 0, 1)
  Match: k=5, H edges=[(1, 2), (1, 3), (2, 3), (3, 4)], c=(0, 0, 1, 1, 1)
  Match: k=5, H edges=[(1, 2), (1, 3), (2, 3), (3, 4)], c=(0, 1, 0, 1, 1)
Checked 52 atlas graphs (n<=atlas_max=5). Found 4 (H,c) pairs.

Done in 68.2s: 4 (H,c) pairs → 2 non-isomorphic unfoldings.

Non-isomorphic unfoldings found: 2
  n=5, H_edges=[(0, 3), (1, 2), (1, 3), (2, 3)], c=(1, 0, 1, 1), iso_to_original=True
  n=6, H_edges=[(1, 2), (1, 3), (2, 3), (3, 4)], c=(0, 0, 1, 1, 1), iso_to_original=False

Original ER graph rec

## Demo 4 — Multi-label training data generation

For the ML pipeline in `unfolding_pipeline.ipynb`, we need a training set where each
example $(r, G)$ pairs a reduction with a **valid admissible unfolding** $G$. Currently
the training data uses only the original graph — but for $n=10$ ER graphs, the algorithm
here cannot enumerate all unfoldings (complement has 9 vertices, far beyond atlas size 7).

**Feasible range with graph atlas (n_atlas_max=7):**
- Original graph $n \leq 8$: complement $k \leq 7$ → full enumeration possible.
- Original graph $n = 9, 10$: complement $k = 8, 9$ → partial enumeration (finds smaller unfoldings $k \leq 7$).

**Workaround for large graphs:** Restrict training to $n \leq 8$, OR use a different approach
for larger graphs (e.g., Newton's method in the space of adjacency matrices).

Below we generate a small batch of multi-label training data for graphs on 5 vertices.


In [9]:
if sp:
    print("Generating multi-label training data for random graphs on n=5...")
    print("(Each row: one reduction r(lam) paired with ALL admissible unfoldings)")
    print("=" * 70)

    n_examples = 5
    seed = 0
    training_data = []

    for trial in range(50):
        if len(training_data) >= n_examples:
            break
        G = nx.erdos_renyi_graph(5, 0.5, seed=seed + trial)
        if not nx.is_connected(G):
            continue

        kept = 0
        r = compute_reduction(G, kept, lam)
        unfoldings = reconstruct_all_unfoldings(
            G, kept, lam, n_atlas_max=5, verbose=False
        )
        if not unfoldings:
            continue

        training_data.append({
            'source_graph_edges': sorted(G.edges()),
            'reduction': str(r),
            'n_unfoldings': len(unfoldings),
            'unfolding_sizes': sorted(set(rec['n'] for rec in unfoldings)),
        })
        print(f"  Example {len(training_data)}: G edges={sorted(G.edges())}")
        print(f"    r = {r}")
        print(f"    {len(unfoldings)} unfoldings, sizes: {sorted(set(rec['n'] for rec in unfoldings))}")

    print(f"\nGenerated {len(training_data)} training examples.")
    print("Multi-label training allows the ML model to predict any admissible unfolding.")

Generating multi-label training data for random graphs on n=5...
(Each row: one reduction r(lam) paired with ALL admissible unfoldings)
  Example 1: G edges=[(0, 3), (0, 4), (1, 3), (2, 3), (2, 4)]
    r = (2*lambda**3 - lambda)/(lambda**4 - 3*lambda**2 + 1)
    2 unfoldings, sizes: [5, 6]
  Example 2: G edges=[(0, 1), (0, 4), (1, 2), (1, 3), (2, 4), (3, 4)]
    r = 2*lambda/(lambda**2 - 4)
    2 unfoldings, sizes: [5, 6]
  Example 3: G edges=[(0, 1), (0, 3), (1, 3), (1, 4), (2, 4), (3, 4)]
    r = (2*lambda**2 - 2)/(lambda**3 - lambda**2 - 3*lambda + 1)
    2 unfoldings, sizes: [5, 6]
  Example 4: G edges=[(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (1, 3), (3, 4)]
    r = (4*lambda + 2)/(lambda**2 - lambda - 1)
    2 unfoldings, sizes: [5, 6]
  Example 5: G edges=[(0, 3), (0, 4), (1, 2), (1, 4), (2, 4)]
    r = (2*lambda**2 - 2*lambda - 2)/(lambda**3 - lambda**2 - 2*lambda)
    2 unfoldings, sizes: [5, 6]

Generated 5 training examples.
Multi-label training allows the ML model to predict

## Notes and Next Steps

### What this notebook demonstrates
- Algebraic reconstruction of **all** admissible unfoldings from a target reduction $r(\lambda)$.
- The approach is **exact**: it finds every graph $G$ (up to the atlas limit) whose reduction
  to the kept vertex equals the target.
- For $P_3$ and $P_5$, the original graphs are correctly recovered among the unfoldings.

### Scaling challenge
- **Graph atlas covers $n \leq 7$ vertices.** For the $n=10$ ER graphs from the ML pipeline,
  the complement has 9 vertices — impossible to enumerate directly.
- **Per-graph cost**: symbolic matrix inverse + $2^k$ quadratic form evaluations.
  For $k=7$: ~5 minutes. For $k=9$: days.

### Connection to multi-label ML training
- For graphs $n \leq 8$, this notebook generates the **ground truth multi-label targets** needed
  to train the ML model with a best-of-many loss.
- The ML model receives a reduction $r$ and predicts an adjacency matrix $A$; training on ALL
  admissible $A$ (not just the source graph) eliminates the false-negative gradient problem.

### Recommended next steps
1. **Retrain the ML pipeline on $n \leq 8$ graphs** using multi-label targets from this notebook.
2. **Extend atlas coverage**: generate all non-isomorphic graphs on $k=8,9$ vertices
   (nauty/networkx can enumerate these, ~12000 graphs for $k=8$).
3. **Faster coupling search**: use spectral filtering to prune $(H, \mathbf{c})$ candidates
   before the expensive symbolic check (e.g., necessary conditions from partial fraction
   decomposition of $r$).
